In [11]:
import os
import time
import re
import pandas as pd
from openai import OpenAI

# =========================
# Configuration
# =========================
WORKING_DATASET = "Raw_digital_banking_reviews.csv"
PROMPT_DATASET = "Prompt_LLaMA_16_Constructs_Actor_Attribution.txt"

OUTPUT_TEXT_FILE = "results_llama_raw_text.txt"
OUTPUT_RAW_CSV_FILE = "results_llama_raw_output.csv"
OUTPUT_PARSED_XLSX_FILE = "LLM_16_constructs_actor_attribution.xlsx"

MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct:novita"

API_TOKEN = "Insert API token here"

# =========================
# Client
# =========================
client = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=API_TOKEN,
)

# =========================
# Load prompt template
# =========================
with open(PROMPT_DATASET, "r", encoding="utf-8", errors="replace") as file:
    prompt_template = file.read()

if "{{number}}" in prompt_template or "{{verbal_report}}" in prompt_template:
    raise ValueError(
        "Prompt file still contains {{number}} or {{verbal_report}}. "
        "Replace them with NUMBER and VERBAL_REPORT."
    )

# =========================
# Load dataset
# =========================
reviews_df = pd.read_csv(WORKING_DATASET, encoding="utf-8")
reviews_df.columns = reviews_df.columns.str.strip().str.lower()

required_columns = {"number", "verbal_report"}
missing_cols = required_columns - set(reviews_df.columns)
if missing_cols:
    raise ValueError(f"Missing required columns in CSV: {missing_cols}")

reviews_df = reviews_df.dropna(subset=["verbal_report"]).copy()
reviews_df["verbal_report"] = reviews_df["verbal_report"].astype(str).str.strip()
reviews_df = reviews_df[reviews_df["verbal_report"] != ""].copy()

# =========================
# Prompt builder
# =========================
def generate_prompt(template: str, number, verbal_report) -> str:
    return (
        template
        .replace("NUMBER", str(number))
        .replace("VERBAL_REPORT", str(verbal_report))
    )

# =========================
# Inference
# =========================
results = []

for idx, row in reviews_df.iterrows():
    number = row["number"]
    verbal_report = row["verbal_report"]
    prompt = generate_prompt(prompt_template, number, verbal_report)

    try:
        completion = client.chat.completions.create(
            model=MODEL_ID,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=5000,
            temperature=0.0,
        )
        response_text = completion.choices[0].message.content

    except Exception as e:
        response_text = f"ERROR for number {number}: {e}"

    results.append({
        "number": number,
        "verbal_report": verbal_report,
        "model_output": response_text,
    })

    print(f"Processed row {idx + 1}/{len(reviews_df)} | number={number}")
    time.sleep(0.5)

# =========================
# Save raw outputs
# =========================
with open(OUTPUT_TEXT_FILE, "w", encoding="utf-8") as f:
    for item in results:
        f.write(str(item["model_output"]).strip())
        f.write("\n\n")

results_df = pd.DataFrame(results)
results_df.to_csv(OUTPUT_RAW_CSV_FILE, index=False, encoding="utf-8")

print(f"Saved raw text output to: {OUTPUT_TEXT_FILE}")
print(f"Saved raw CSV output to: {OUTPUT_RAW_CSV_FILE}")

# =========================
# Parse model output
# =========================
with open(OUTPUT_TEXT_FILE, "r", encoding="utf-8") as f:
    raw_text = f.read()

sections = re.split(r"(?=##OBSNR#)", raw_text)
sections = [s.strip() for s in sections if s.strip()]

data = []

obsnr_pattern = re.compile(r"##OBSNR#\s*([^\n:]+)\s*:?", re.IGNORECASE)

header_pattern = re.compile(
    r"\*\*\s*(?:(.*?)\s*-\s*)?([A-Za-z][A-Za-z0-9 ]*)\s*\*\*",
    re.IGNORECASE
)

construct_to_category = {
    "PerformanceExpectancy": "Service Quality",
    "EffortExpectancy": "Service Quality",
    "ServiceSupport": "Service Quality",
    "PriceValue": "Service Quality",
    "DataPrivacyRisk": "Operational Risk",
    "FinancialRisk": "Operational Risk",
    "SystemPerformanceRisk": "Operational Risk",
    "UserAnxietyRisk": "Operational Risk",
    "ServiceDelayRisk": "Operational Risk",
    "Trust": "Customer Experience",
    "HedonicMotivation": "Customer Experience",
    "SocialInfluence": "Customer Experience",
    "UsageHabit": "Customer Experience",
    "UsageExperience": "Customer Behavioral Outcomes",
    "BehavioralIntention": "Customer Behavioral Outcomes",
    "RecommendationIntention": "Customer Behavioral Outcomes",
}

def grab(pattern, text):
    m = pattern.search(text)
    return re.sub(r"\s+", " ", m.group(1)).strip() if m else None

verbal_pattern = re.compile(r"Verbal report:\s*(.*?)(?=\nOriginal statement:|\Z)", re.I | re.S)
orig_pattern = re.compile(r"Original statement:\s*(.*?)(?=\nReasoning:|\Z)", re.I | re.S)
reason_pattern = re.compile(r"Reasoning:\s*(.*?)(?=\nGrade:|\nActor attribution:|\nAttribution valence:|\Z)", re.I | re.S)
grade_pattern = re.compile(r"Grade\s*:\s*(?:@@)?\s*([-+]?\d{1,3}|999)\s*(?:@@)?", re.I)
actor_pattern = re.compile(r"Actor attribution:\s*(.*?)(?=\nAttribution valence:|\Z)", re.I | re.S)
valence_pattern = re.compile(r"Attribution valence:\s*(.*?)(?=\nAttribution reasoning:|\Z)", re.I | re.S)
attr_reason_pattern = re.compile(r"Attribution reasoning:\s*(.*)", re.I | re.S)

for sec_idx, section in enumerate(sections):
    obsnr = grab(obsnr_pattern, section)
    h = header_pattern.search(section)

    if not h:
        print("No header found in section:", section[:300])
        continue

    raw_category = h.group(1).strip() if h.group(1) else None
    keyword = h.group(2).strip().replace(" ", "")

    if keyword.lower() == "actorattribution":
        continue

    category = raw_category if raw_category else construct_to_category.get(keyword, "Unknown")

    block = section[h.end():].strip()

    grade_match = grade_pattern.search(block)
    grade = grade_match.group(1).strip() if grade_match else None

    data.append({
        "OBSR ID": obsnr,
        "Category": category,
        "Keyword": keyword,
        "Verbal Report": grab(verbal_pattern, block),
        "Original Statement": grab(orig_pattern, block),
        "Reasoning": grab(reason_pattern, block),
        "Grade": grade,
        "Actor Attribution": grab(actor_pattern, block),
        "Attribution Valence": grab(valence_pattern, block),
        "Attribution Reasoning": grab(attr_reason_pattern, block),
    })

parsed_df = pd.DataFrame(data)

print("Rows extracted:", len(parsed_df))
print(parsed_df.head(20))

parsed_df.to_excel(OUTPUT_PARSED_XLSX_FILE, index=False)
print(f"FINAL FILE SAVED: {OUTPUT_PARSED_XLSX_FILE}")

Processed row 1/200 | number=501
Processed row 2/200 | number=502
Processed row 3/200 | number=503
Processed row 4/200 | number=504
Processed row 5/200 | number=505
Processed row 6/200 | number=506
Processed row 7/200 | number=507
Processed row 8/200 | number=508
Processed row 9/200 | number=509
Processed row 10/200 | number=510
Processed row 11/200 | number=511
Processed row 12/200 | number=512
Processed row 13/200 | number=513
Processed row 14/200 | number=514
Processed row 15/200 | number=515
Processed row 16/200 | number=516
Processed row 17/200 | number=517
Processed row 18/200 | number=518
Processed row 19/200 | number=519
Processed row 20/200 | number=520
Processed row 21/200 | number=521
Processed row 22/200 | number=522
Processed row 23/200 | number=523
Processed row 24/200 | number=524
Processed row 25/200 | number=525
Processed row 26/200 | number=526
Processed row 27/200 | number=527
Processed row 28/200 | number=528
Processed row 29/200 | number=529
Processed row 30/200 | 